In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import os
import sys
from typing import List

sys.path.append('/Users/harendrakumar/Documents/bank_loan_prediction/')

In [3]:
from ML_Pipelines.ml.utils.data_loader import get_latest_partition_data
from ML_Pipelines.ml.pipelines.ingestion import IngestionPipeline
from ML_Pipelines.ml.pipelines import transformation
from ML_Pipelines.ml.utils import databalancer, data_loader

In [4]:
from ML_Pipelines.ml.utils.utility import (
    drop_columns_having_nulls_above_threshold,
    fill_numeric_missing_values,
    fill_categorical_missing_values,
    fill_numeric_missing_values_using_interpolation,
    drop_columns_with_IDs,
    encode_cat_columns
)

In [5]:
steps = [drop_columns_with_IDs,
         drop_columns_having_nulls_above_threshold, 
         fill_numeric_missing_values, 
         fill_categorical_missing_values,
        encode_cat_columns]

In [6]:
data_dir="../../data_folder/train"

In [7]:
df = transformation.TransformationPipeline(ingestion_pipeline=IngestionPipeline(data_dir=data_dir),
                                     transformation_steps=steps).run_transformation()

INFO:ML_Pipelines.ml.pipelines.ingestion:Latest partition file ../../data_folder/train/credit_train_20241027.csv loaded successfully.
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.pipelines.transformation:Applying transformation step: drop_columns_with_IDs
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.utils.utility:Dropping ID columns: ['Loan ID', 'Customer ID']
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.pipelines.transformation:Applying transformation step: drop_columns_having_nulls_above_threshold
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.utils.utility:Dropping columns with more than 50.0% null values.
INFO:ML_Pipelines.ml.utils.utility:Dropping columns with null percentage above 50.0: ['Months since last delinquent']
INFO:ML_Pipelines.ml.pipelines.transformati

In [8]:
df.head()

,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,1,445412.0,1,709.000000,1.167493e+06,8,1,5,5214.74,17.2,6.0,1.0,228190.0,416746.0,1.0,0.0
1,1,262328.0,1,1076.456089,1.378277e+06,1,1,3,33295.98,21.1,35.0,0.0,229976.0,850784.0,0.0,0.0
2,1,99999999.0,1,741.000000,2.231892e+06,8,2,3,29200.53,14.9,18.0,1.0,297996.0,750090.0,0.0,0.0
3,1,347666.0,0,721.000000,8.069490e+05,3,2,3,8741.90,12.0,9.0,0.0,256329.0,386958.0,0.0,0.0
4,1,176220.0,1,1076.456089,1.378277e+06,5,3,3,20639.70,6.1,15.0,0.0,253460.0,427174.0,0.0,0.0


In [9]:
df.shape

(100000, 16)

In [10]:
df.columns[df.columns.str.contains("ID")].tolist()

[]

In [11]:
df.columns

Index(['Loan Status', 'Current Loan Amount', 'Term', 'Credit Score',
       'Annual Income', 'Years in current job', 'Home Ownership', 'Purpose',
       'Monthly Debt', 'Years of Credit History', 'Number of Open Accounts',
       'Number of Credit Problems', 'Current Credit Balance',
       'Maximum Open Credit', 'Bankruptcies', 'Tax Liens'],
      dtype='object')

In [12]:
balanced_df = databalancer.DataBalancer(data=df, target_column='Loan Status').balance_data()

🤖 Auto-selected method based on imbalance ratio (0.29): UNDER
✅ Balancing done using: UNDER
New class distribution:
Counter({0: 22639, 1: 22639})


In [13]:
balanced_df.head(5)

,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens,Loan Status
5,206602.0,1,7290.000000,8.968570e+05,1,1,3,16367.74,17.3,6.0,0.0,215308.0,272448.0,0.0,0.0,0
7,648714.0,0,1076.456089,1.378277e+06,10,1,1,14806.13,8.2,15.0,0.0,193306.0,864204.0,0.0,0.0,0
16,653004.0,0,1076.456089,1.378277e+06,7,1,3,14537.09,20.5,9.0,0.0,302309.0,413754.0,0.0,0.0,0
20,317108.0,0,687.000000,1.133274e+06,8,3,3,9632.81,17.4,4.0,0.0,60287.0,126940.0,0.0,0.0,0
22,153252.0,1,714.000000,1.890690e+06,2,3,3,21900.35,15.7,12.0,0.0,891594.0,1081014.0,0.0,0.0,0


In [14]:
balanced_df['Loan Status'].value_counts()

Loan Status
0    22639
1    22639
Name: count, dtype: int64